# Etapa 1: Selección y caracterización del dataset NYC TLC Yellow Taxi

En esta etapa se caracteriza el dataset de viajes en taxi amarillo de la ciudad de Nueva York durante los años 2024 y 2025, publicado por la New York City Taxi and Limousine Commission (TLC). El dataset contiene registros transaccionales de viajes con atributos de fechas, distancias, tarifas y zonas de origen y destino, lo cual lo hace adecuado para el análisis con PySpark en un entorno de Big Data.

Se utilizan dos años completos (2024 y 2025) para garantizar un volumen total superior a 1 GB y para habilitar análisis comparativo interanual. El año 2026, parcialmente publicado al momento de este trabajo, se reserva como conjunto de validación temporal para las etapas posteriores del proyecto.

El notebook está pensado para ejecutarse de forma portable, tanto localmente como en Google Colab. Todas las rutas de datos son relativas al notebook (`./data/raw`), por lo que cualquier integrante del equipo puede clonar el repositorio y ejecutar las celdas sin ajustes adicionales.

## 1. Descarga de datos

Los archivos se obtienen directamente del CDN oficial del TLC en formato Parquet. Desde 2022 el TLC distribuye los registros de viajes en Parquet de manera nativa, por su mejor compresión y lectura columnar respecto a CSV. La descarga se realiza de forma idempotente: si el archivo ya existe en disco, se omite, lo cual permite re-ejecutar el notebook sin volver a bajar los datos.

In [5]:
from pathlib import Path
import subprocess

CDN_BASE = "https://d37ci6vzurychx.cloudfront.net"
DATA_DIR = Path("data/raw")
YEARS = [2024, 2025]
LOOKUP_FILE = "taxi_zone_lookup.csv"

In [6]:
def download_if_missing(download_url, target_path):
    """Descarga `download_url` a `target_path` solo si `target_path` no existe.

    Devuelve un string con el estado: 'skip', 'ok' o 'error: <mensaje>'.
    """
    if target_path.exists():
        return "skip"

    target_path.parent.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["curl", "-sSL", "-o", str(target_path), download_url],
        capture_output=True,
        timeout=900,
    )

    if result.returncode != 0:
        return f"error: curl exit {result.returncode}"

    return "ok"

In [7]:
# Parquets mensuales de viajes
for year in YEARS:
    for month in range(1, 13):
        filename = f"yellow_tripdata_{year}-{month:02d}.parquet"
        url = f"{CDN_BASE}/trip-data/{filename}"
        target = DATA_DIR / filename
        status = download_if_missing(url, target)
        print(f"{status:>6}  {filename}")

# Tabla de referencia de zonas de taxi
url = f"{CDN_BASE}/misc/{LOOKUP_FILE}"
target = DATA_DIR / LOOKUP_FILE
status = download_if_missing(url, target)
print(f"{status:>6}  {LOOKUP_FILE}")

    ok  yellow_tripdata_2024-01.parquet
    ok  yellow_tripdata_2024-02.parquet
    ok  yellow_tripdata_2024-03.parquet
    ok  yellow_tripdata_2024-04.parquet
    ok  yellow_tripdata_2024-05.parquet
    ok  yellow_tripdata_2024-06.parquet
    ok  yellow_tripdata_2024-07.parquet
    ok  yellow_tripdata_2024-08.parquet
    ok  yellow_tripdata_2024-09.parquet
    ok  yellow_tripdata_2024-10.parquet
    ok  yellow_tripdata_2024-11.parquet
    ok  yellow_tripdata_2024-12.parquet
    ok  yellow_tripdata_2025-01.parquet
    ok  yellow_tripdata_2025-02.parquet
    ok  yellow_tripdata_2025-03.parquet
    ok  yellow_tripdata_2025-04.parquet
    ok  yellow_tripdata_2025-05.parquet
    ok  yellow_tripdata_2025-06.parquet
    ok  yellow_tripdata_2025-07.parquet
    ok  yellow_tripdata_2025-08.parquet
    ok  yellow_tripdata_2025-09.parquet
    ok  yellow_tripdata_2025-10.parquet
    ok  yellow_tripdata_2025-11.parquet
    ok  yellow_tripdata_2025-12.parquet
    ok  taxi_zone_lookup.csv


## 2. Resumen de archivos descargados

Se reporta el inventario y el tamaño en disco. La rúbrica del curso pide un dataset por encima de 1 GB. El formato Parquet ya está comprimido, por lo que el tamaño en disco es menor al equivalente en CSV pero conserva la totalidad de los registros.

In [8]:
files = sorted(DATA_DIR.glob("*"))
total_bytes = sum(f.stat().st_size for f in files)

for f in files:
    size_mb = f.stat().st_size / (1024 ** 2)
    print(f"{size_mb:>8.1f} MB   {f.name}")

print()
print(f"Archivos: {len(files)} (esperados: {len(YEARS) * 12 + 1})")
print(f"Tamaño total: {total_bytes / (1024 ** 3):.2f} GB")

parquets = [f for f in files if f.suffix == ".parquet"]
if parquets:
    avg_mb = sum(f.stat().st_size for f in parquets) / len(parquets) / (1024 ** 2)
    print(f"Tamaño promedio por Parquet: {avg_mb:.1f} MB")

     0.0 MB   taxi_zone_lookup.csv
    47.6 MB   yellow_tripdata_2024-01.parquet
    48.0 MB   yellow_tripdata_2024-02.parquet
    57.3 MB   yellow_tripdata_2024-03.parquet
    56.4 MB   yellow_tripdata_2024-04.parquet
    59.7 MB   yellow_tripdata_2024-05.parquet
    57.1 MB   yellow_tripdata_2024-06.parquet
    49.9 MB   yellow_tripdata_2024-07.parquet
    48.7 MB   yellow_tripdata_2024-08.parquet
    58.3 MB   yellow_tripdata_2024-09.parquet
    61.4 MB   yellow_tripdata_2024-10.parquet
    57.8 MB   yellow_tripdata_2024-11.parquet
    58.7 MB   yellow_tripdata_2024-12.parquet
    56.4 MB   yellow_tripdata_2025-01.parquet
    57.5 MB   yellow_tripdata_2025-02.parquet
    66.7 MB   yellow_tripdata_2025-03.parquet
    64.2 MB   yellow_tripdata_2025-04.parquet
    74.2 MB   yellow_tripdata_2025-05.parquet
    70.1 MB   yellow_tripdata_2025-06.parquet
    63.8 MB   yellow_tripdata_2025-07.parquet
    59.4 MB   yellow_tripdata_2025-08.parquet
    69.1 MB   yellow_tripdata_2025-09.parquet